# Info 2950 Final Project Phase 2

## Research Questions

Question: Can we accurately predict the daily bike rental sharing count based on the environemntal conditions (weather, temp, hum, windspeed) and if that day is holiday or not.

In this analysis, we aim to explore correlations between daily bike rental counts and various factors, such as weather conditions and holidays. Specifically, we will investigate the relationships between temperature, humidity, wind speed, weather conditions and holiday. We will train a multivariable regression model to see if we can reliably predict the number of daily bike rentals based on these environmental and holiday conditions.

## Data Description
This dataset contains the daily count of rental bikes between 2011 and 2012 in the Capital bikeshare system with the corresponding weather and seasonal information.

## Data Cleaning

We downloaded the data from UC Irvine Machine Learning Repository in the form of a .csv file. To perform data cleaning, we placed the .csv file in the same folder as our Phase 2, Our first steps are to load the dataset and check for null values.


In [20]:
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import duckdb

In [21]:
# load data
data = pd.read_csv("hour.csv")
data.head()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


Since we are only interested in the total count of daily bike rentals, we are going to delete the causal and registered user count columns. 


In [22]:
# remove "casual", "registered" column from the data
data = data.drop(data.columns[[-3, -2]], axis=1)
data.head()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,1


The description for the workingday column is “if the day is neither weekend nor holiday is 1, otherwise is 0.” Since the workingday column and the weekday and holiday columns are mostly based off of each other, we are going to delete the weekday and holiday columns to simplify the regression model. We do not want a regression model that is too complex and has high variance.

In [23]:
# remove the holiday, weekend column
data = data.drop(['holiday', 'weekday'], axis=1)
data.head()

,instant,dteday,season,yr,mnth,hr,workingday,weathersit,temp,atemp,hum,windspeed,cnt
0,1,2011-01-01,1,0,1,0,0,1,0.24,0.2879,0.81,0.0,16
1,2,2011-01-01,1,0,1,1,0,1,0.22,0.2727,0.80,0.0,40
2,3,2011-01-01,1,0,1,2,0,1,0.22,0.2727,0.80,0.0,32
3,4,2011-01-01,1,0,1,3,0,1,0.24,0.2879,0.75,0.0,13
4,5,2011-01-01,1,0,1,4,0,1,0.24,0.2879,0.75,0.0,1


We find that the dataset contains a column "instant" that works as an index column. We decided to remove this column.

In [24]:
# drop the first column "instant"
data = data.drop(['instant'], axis=1)
data.head()

,dteday,season,yr,mnth,hr,workingday,weathersit,temp,atemp,hum,windspeed,cnt
0,2011-01-01,1,0,1,0,0,1,0.24,0.2879,0.81,0.0,16
1,2011-01-01,1,0,1,1,0,1,0.22,0.2727,0.80,0.0,40
2,2011-01-01,1,0,1,2,0,1,0.22,0.2727,0.80,0.0,32
3,2011-01-01,1,0,1,3,0,1,0.24,0.2879,0.75,0.0,13
4,2011-01-01,1,0,1,4,0,1,0.24,0.2879,0.75,0.0,1


We decided to divide the "hour" column into four groups:
- first group: hour 0-5 (dawn, 1)
- second group: hour 6-11 (morning, 2)
- third group: hour 12-17 (afternoon, 3)
- fourth group: hour: 18-23 (night, 4)

We realized that the "weathersit" variable may differ among this period of time. So we dicided to choose the majority weather situation.